# Exploring Stock Data using brapi-dev api

## Load ingested data into spark tables

In [1]:
from spark import loader

spark = loader.create_spark_session("brapi-api")

loader.load_spark_tables(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 19:56:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/06 19:56:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Fundamental Analysis Metrics

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

### Companies with Consistent Profits

In [3]:
# Create a window to grab the most recent 4 quarters per stock
window_spec = Window.partitionBy("stock").orderBy(F.col("endDate").desc())

consistent_profits_df = (
    spark.table("incomestatementhistoryquarterly")
    # Rank quarters by date
    .withColumn("rank", F.row_number().over(window_spec))
    # Filter for the last year of data (last 4 quarters)
    .filter(F.col("rank") <= 4)
    # Check if the company was profitable in each quarter (1 if true, 0 if false)
    .withColumn("is_profitable", F.when(F.col("netIncome") > 0, 1).otherwise(0))
    # Aggregate by stock to see how many of the 4 quarters were profitable
    .groupBy("stock")
    .agg(
        F.sum("is_profitable").alias("profitable_quarters_count"),
        F.avg("netIncome").alias("avg_quarterly_net_income")
    )
    # Target companies that hit 4 out of 4 profitable quarters
    .filter(F.col("profitable_quarters_count") == 4)
    .select("stock", "avg_quarterly_net_income")
)

consistent_profits_df.toPandas().head()

,stock,avg_quarterly_net_income
0,ABCB4,2.002132e+08
1,ABEV3,4.017338e+09
2,AFLT3,8.214500e+06
3,ALLD3,8.558450e+07
4,ALOS3,2.335350e+08


### Companies with Low Debt in Proportion to EBITDA

In [4]:
# Window spec to get the absolute latest recorded quarter
latest_window = Window.partitionBy("stock").orderBy(F.col("endDate").desc())

low_debt_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    # Grab the most recent quarter
    .filter(F.col("rank") == 1)
    # Avoid division by zero or working with companies with negative EBITDA
    .filter(F.col("ebitda") > 0)
    # Annualizing quarterly EBITDA (EBITDA * 4) to accurately measure Debt/EBITDA
    .withColumn("annualized_ebitda", F.col("ebitda") * 4)
    .withColumn("debt_to_ebitda", F.col("totalDebt") / F.col("annualized_ebitda"))
    # Filter for companies with a conservative ratio (e.g., Debt/EBITDA < 2.0)
    .filter(F.col("debt_to_ebitda") < 2.0)
    .select("stock", "totalDebt", "ebitda", "debt_to_ebitda")
    .orderBy("debt_to_ebitda")
)

low_debt_df.toPandas().head()

,stock,totalDebt,ebitda,debt_to_ebitda
0,BRAP3,493000,1828846000,0.000067
1,BRAP4,493000,1828846000,0.000067
2,CXSE3,13958000,4748332000,0.000735
3,BAUH4,5208000,29461000,0.044194
4,CAMB3,16822000,79254000,0.053064


### Companies with Consistency Growth

In [5]:
# Window spec to pull the last 4 quarters
growth_window = Window.partitionBy("stock").orderBy(F.col("endDate").desc())

consistent_growth_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(growth_window))
    .filter(F.col("rank") <= 4)
    # A quarter is marked '1' if growth was positive (e.g., > 5% YoY growth)
    .withColumn("is_growing", F.when(F.col("revenueGrowth") > 0.05, 1).otherwise(0))
    .groupBy("stock")
    .agg(
        F.sum("is_growing").alias("consistent_growth_quarters"),
        F.avg("revenueGrowth").alias("avg_revenue_growth")
    )
    # Keep companies that hit our target growth benchmark in all 4 quarters
    .filter(F.col("consistent_growth_quarters") == 4)
    .select("stock", "avg_revenue_growth")
    .orderBy(F.col("avg_revenue_growth").desc())
)

consistent_growth_df.toPandas().head()

,stock,avg_revenue_growth
0,PINE3,3.756361
1,PINE4,3.756361
2,CALI3,2.871413
3,INBR32,2.275410
4,HBRE3,1.282000


### Return on Equity (ROE) & Return on Assets (ROA)

In [29]:
# Ensure you are checking the latest quarter's efficiency
efficiency_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    # Filters out highly distorted or negative equity anomalies
    .filter((F.col("returnOnEquity") > 0.15) & (F.col("returnOnAssets") > 0.05))
    .select("stock", "returnOnEquity", "returnOnAssets")
    .orderBy("returnOnEquity", ascending=False)
)

efficiency_df.toPandas().head()

,stock,returnOnEquity,returnOnAssets
0,EQPA3,3.457672,0.970171
1,MNPR3,2.568387,1.150877
2,CGAS3,0.894066,0.093325
3,CGAS5,0.894066,0.093325
4,MGEL4,0.840165,0.057048


### Free Cash Flow (FCF) Margin

In [17]:
fcf_health_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    .filter(F.col("totalRevenue") > 0)
    # FCF Margin = Free Cash Flow / Total Revenue
    .withColumn("fcf_margin", F.col("freeCashflow") / F.col("totalRevenue"))
    .filter(F.col("fcf_margin") > 0.10)
    .select("stock", "freeCashflow", "fcf_margin")
    .orderBy("fcf_margin", ascending=False)
)

fcf_health_df.toPandas().head()

,stock,freeCashflow,fcf_margin
0,INEP3,58761000,6.791609
1,INEP4,58761000,6.791609
2,SYNE3,834266000,2.557239
3,ARND3,156648990,2.270440
4,LOGG3,363735000,1.401633


### PEG Ratio (Price/Earnings-to-Growth)

In [18]:
growth_valuation_df = (
    spark.table("defaultkeystatisticshistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    # Look for companies whose stock price isn't outrunning their earnings speed
    .filter((F.col("pegRatio") > 0) & (F.col("pegRatio") <= 1.0))
    .select("stock", "pegRatio", "trailingPE")
    .orderBy("pegRatio", ascending=False)
)

growth_valuation_df.toPandas().head()

,stock,pegRatio,trailingPE
0,CEEB3,0.996561,6.583204
1,CEEB5,0.996561,6.583204
2,ITUB3,0.950632,9.567522
3,ITUB4,0.950632,9.567522
4,ITSA3,0.877418,10.182830


### Short-Term Liquidity: The Current Ratio

In [9]:
liquidity_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    .filter(F.col("currentRatio") >= 1.5)
    .select("stock", "currentRatio", "quickRatio")
)

liquidity_df.toPandas().head()

,stock,currentRatio,quickRatio
0,AFLT3,2.393700,2.327583
1,AGXY3,2.187290,1.799394
2,ALLD3,1.741023,1.225993
3,ALOS3,2.813016,2.813016
4,ALPA3,1.727114,1.276044


### Good Stock Opportunities

In [34]:
stock_opportunities_df = (
    # Baseline active tickers
    spark.table("activetickers").select("stock", "name", "sector")
    # 1. Profits & Growth checks from previous queries
    .join(consistent_profits_df, on="stock", how="inner")
    .join(consistent_growth_df, on="stock", how="inner")
    # 2. Add Long-term Leverage safety
    .join(low_debt_df.select("stock", "debt_to_ebitda"), on="stock", how="inner")
    # 3. Add short-term liquidity safety
    .join(liquidity_df, on="stock", how="inner")
    # 4. Add Capital Efficiency 
    .join(efficiency_df, on="stock", how="inner")
    # 5. Add FCF Quality Check
    .join(fcf_health_df, on="stock", how="inner")
    # 6. Add Growth-Adjusted Valuation
    # .join(growth_valuation_df, on="stock", how="inner")
    .orderBy("avg_quarterly_net_income", ascending=False)
)

stock_opportunities_df.toPandas().head(20)

,stock,name,sector,avg_quarterly_net_income,avg_revenue_growth,debt_to_ebitda,currentRatio,quickRatio,returnOnEquity,returnOnAssets,freeCashflow,fcf_margin
0,TAEE11,TRANSMISSORA ALIANÇA DE ENERGIA ELÉTRICA S.A.,Industrial Services,392087010.0,0.189733,1.330357,2.291451,2.291451,0.197048,0.067402,1528834000,0.326773
1,TAEE3,TRANSMISSORA ALIANÇA DE ENERGIA ELÉTRICA S.A.,Industrial Services,392087010.0,0.189733,1.330357,2.291451,2.291451,0.197048,0.067402,1528834000,0.326773
2,TAEE4,TRANSMISSORA ALIANÇA DE ENERGIA ELÉTRICA S.A.,Industrial Services,392087010.0,0.189733,1.330357,2.291451,2.291451,0.197048,0.067402,1528834000,0.326773
3,RIAA3,RIAA3,Distribution Services,376713750.0,0.096295,0.299312,1.595530,1.276005,0.283291,0.106910,1877788900,0.176916
4,MNPR3,MINUPAR PARTICIPACOES S.A.,Process Industries,129064000.0,0.236585,0.207758,2.905504,2.577272,2.568387,1.150877,113758000,0.253941
5,CSED3,CRUZEIRO DO SUL EDUCACIONAL S.A.,Consumer Services,67984745.0,0.097128,0.765416,1.548665,1.548665,0.164598,0.055949,585050000,0.204108
6,SEER3,SER EDUCACIONAL S.A.,Consumer Services,61676500.0,0.114359,0.585428,1.558467,1.558467,0.174032,0.069145,231722000,0.102518


## Full Data Catalog

In [11]:
for table in spark.catalog.listTables():
    print(f"---- {table.name} ----\n")
    spark.table(table.name).printSchema()

---- activetickers ----

root
 |-- change: double (nullable = true)
 |-- close: double (nullable = true)
 |-- logo: string (nullable = true)
 |-- market_cap: double (nullable = true)
 |-- name: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- stock: string (nullable = true)
 |-- subType: string (nullable = true)
 |-- type: string (nullable = true)
 |-- volume: long (nullable = true)

---- balancesheethistoryquarterly ----

root
 |-- stock: string (nullable = true)
 |-- type: string (nullable = true)
 |-- endDate: string (nullable = true)
 |-- cash: long (nullable = true)
 |-- shortTermInvestments: long (nullable = true)
 |-- netReceivables: long (nullable = true)
 |-- inventory: long (nullable = true)
 |-- otherCurrentAssets: long (nullable = true)
 |-- totalCurrentAssets: long (nullable = true)
 |-- longTermInvestments: long (nullable = true)
 |-- propertyPlantEquipment: long (nullable = true)
 |-- otherAssets: long (nullable = true)
 |-- totalAssets: long (nullable